In [2]:
import pandas as pd
import sqlite3
import numpy as np

# Joins en SQL

## Ejemplo: viajes en taxi en Nueva York

Fuente: https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page

In [4]:
# Base de datos de viajes en enero de 2025 en Nueva York

# El formato Parquet es común para tablas grandes, guarda los datos por columna y es más fácil de comprimir.
trips = pd.read_parquet("yellow_tripdata_2025-01.parquet")
trips

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,1,2025-01-01 00:18:38,2025-01-01 00:26:59,1.0,1.60,1.0,N,229,237,1,10.00,3.5,0.5,3.00,0.0,1.0,18.00,2.5,0.0,0.00
1,1,2025-01-01 00:32:40,2025-01-01 00:35:13,1.0,0.50,1.0,N,236,237,1,5.10,3.5,0.5,2.02,0.0,1.0,12.12,2.5,0.0,0.00
2,1,2025-01-01 00:44:04,2025-01-01 00:46:01,1.0,0.60,1.0,N,141,141,1,5.10,3.5,0.5,2.00,0.0,1.0,12.10,2.5,0.0,0.00
3,2,2025-01-01 00:14:27,2025-01-01 00:20:01,3.0,0.52,1.0,N,244,244,2,7.20,1.0,0.5,0.00,0.0,1.0,9.70,0.0,0.0,0.00
4,2,2025-01-01 00:21:34,2025-01-01 00:25:06,3.0,0.66,1.0,N,244,116,2,5.80,1.0,0.5,0.00,0.0,1.0,8.30,0.0,0.0,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3475221,2,2025-01-31 23:01:48,2025-01-31 23:16:29,NaN,3.35,NaN,None,79,237,0,15.85,0.0,0.5,0.00,0.0,1.0,20.60,NaN,NaN,0.75
3475222,2,2025-01-31 23:50:29,2025-02-01 00:17:27,NaN,8.73,NaN,None,161,116,0,28.14,0.0,0.5,0.00,0.0,1.0,32.89,NaN,NaN,0.75
3475223,2,2025-01-31 23:26:59,2025-01-31 23:43:01,NaN,2.64,NaN,None,144,246,0,14.91,0.0,0.5,0.00,0.0,1.0,19.66,NaN,NaN,0.75
3475224,2,2025-01-31 23:14:34,2025-01-31 23:34:52,NaN,3.16,NaN,None,142,107,0,17.55,0.0,0.5,0.00,0.0,1.0,22.30,NaN,NaN,0.75


La base contiene ID del lugar de Pick-up (subida) e ID del lugar de Drop-Off (bajada).

La información sobre cada ID la tenemos en una tabla de zonas.

In [5]:
zones = pd.read_csv("taxi_zone_lookup.csv")
zones

,LocationID,Borough,Zone,service_zone
0,1,EWR,Newark Airport,EWR
1,2,Queens,Jamaica Bay,Boro Zone
2,3,Bronx,Allerton/Pelham Gardens,Boro Zone
3,4,Manhattan,Alphabet City,Yellow Zone
4,5,Staten Island,Arden Heights,Boro Zone
...,...,...,...,...
260,261,Manhattan,World Trade Center,Yellow Zone
261,262,Manhattan,Yorkville East,Yellow Zone
262,263,Manhattan,Yorkville West,Yellow Zone
263,264,Unknown,NaN,NaN


In [6]:
# Vemos los barrios
zones.Borough.value_counts()

Borough
Queens           69
Manhattan        69
Brooklyn         61
Bronx            43
Staten Island    20
EWR               1
Unknown           1
Name: count, dtype: int64

In [7]:
# Para no trabajar con tantos datos nos quedamos solo con los viajes del 1 de enero
trips = trips[trips.tpep_pickup_datetime.dt.day == 1]
trips

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,1,2025-01-01 00:18:38,2025-01-01 00:26:59,1.0,1.60,1.0,N,229,237,1,10.00,3.5,0.5,3.00,0.00,1.0,18.00,2.5,0.0,0.0
1,1,2025-01-01 00:32:40,2025-01-01 00:35:13,1.0,0.50,1.0,N,236,237,1,5.10,3.5,0.5,2.02,0.00,1.0,12.12,2.5,0.0,0.0
2,1,2025-01-01 00:44:04,2025-01-01 00:46:01,1.0,0.60,1.0,N,141,141,1,5.10,3.5,0.5,2.00,0.00,1.0,12.10,2.5,0.0,0.0
3,2,2025-01-01 00:14:27,2025-01-01 00:20:01,3.0,0.52,1.0,N,244,244,2,7.20,1.0,0.5,0.00,0.00,1.0,9.70,0.0,0.0,0.0
4,2,2025-01-01 00:21:34,2025-01-01 00:25:06,3.0,0.66,1.0,N,244,116,2,5.80,1.0,0.5,0.00,0.00,1.0,8.30,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2950896,2,2025-01-01 23:41:33,2025-01-01 23:58:46,NaN,6.09,NaN,None,197,36,0,2.94,0.0,0.5,0.00,0.00,1.0,4.44,NaN,NaN,0.0
2950897,2,2025-01-01 23:02:52,2025-01-01 23:08:01,NaN,0.97,NaN,None,170,234,0,-0.71,0.0,0.5,0.00,0.00,1.0,3.29,NaN,NaN,0.0
2950898,2,2025-01-01 23:33:52,2025-01-01 23:48:39,NaN,6.00,NaN,None,219,86,0,-3.34,0.0,0.5,0.00,0.00,1.0,-1.84,NaN,NaN,0.0
2950899,2,2025-01-01 23:34:59,2025-01-01 23:51:28,NaN,3.59,NaN,None,215,258,0,-0.94,0.0,0.5,0.00,0.00,1.0,3.06,NaN,NaN,0.0


In [8]:
trips.info()

<class 'pandas.core.frame.DataFrame'>
Index: 90189 entries, 0 to 2950900
Data columns (total 20 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   VendorID               90189 non-null  int32         
 1   tpep_pickup_datetime   90189 non-null  datetime64[us]
 2   tpep_dropoff_datetime  90189 non-null  datetime64[us]
 3   passenger_count        74365 non-null  float64       
 4   trip_distance          90189 non-null  float64       
 5   RatecodeID             74365 non-null  float64       
 6   store_and_fwd_flag     74365 non-null  object        
 7   PULocationID           90189 non-null  int32         
 8   DOLocationID           90189 non-null  int32         
 9   payment_type           90189 non-null  int64         
 10  fare_amount            90189 non-null  float64       
 11  extra                  90189 non-null  float64       
 12  mta_tax                90189 non-null  float64       
 13  tip_

In [9]:
# Pasamos las tablas a una base de datos SQL
con = sqlite3.connect(":memory:")

trips.to_sql("trips", con, index=False, if_exists="replace")
zones.to_sql("zones", con, index=False, if_exists="replace")

265

## Sintaxis básica de `LEFT JOIN`

```sql
SELECT
    tabla1.columna1,
    tabla2.columna2
FROM tabla1
LEFT JOIN tabla2
    ON tabla1.clave = tabla2.clave

### Ejercicio 1

Generar una tabla incorporando a la tabla de viajes una columna con la información del barrio de partida (sin perder ninguna fila)

In [11]:
trips_con_barrio = pd.read_sql_query("""
SELECT
   
FROM 
LEFT JOIN 
    ON 
""", con)
trips_con_barrio

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,...,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,Borough
0,1,2025-01-01 00:18:38,2025-01-01 00:26:59,1.0,1.60,1.0,N,229,237,1,...,3.5,0.5,3.00,0.00,1.0,18.00,2.5,0.0,0.0,Manhattan
1,1,2025-01-01 00:32:40,2025-01-01 00:35:13,1.0,0.50,1.0,N,236,237,1,...,3.5,0.5,2.02,0.00,1.0,12.12,2.5,0.0,0.0,Manhattan
2,1,2025-01-01 00:44:04,2025-01-01 00:46:01,1.0,0.60,1.0,N,141,141,1,...,3.5,0.5,2.00,0.00,1.0,12.10,2.5,0.0,0.0,Manhattan
3,2,2025-01-01 00:14:27,2025-01-01 00:20:01,3.0,0.52,1.0,N,244,244,2,...,1.0,0.5,0.00,0.00,1.0,9.70,0.0,0.0,0.0,Manhattan
4,2,2025-01-01 00:21:34,2025-01-01 00:25:06,3.0,0.66,1.0,N,244,116,2,...,1.0,0.5,0.00,0.00,1.0,8.30,0.0,0.0,0.0,Manhattan
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
90184,2,2025-01-01 23:41:33,2025-01-01 23:58:46,NaN,6.09,NaN,None,197,36,0,...,0.0,0.5,0.00,0.00,1.0,4.44,NaN,NaN,0.0,Queens
90185,2,2025-01-01 23:02:52,2025-01-01 23:08:01,NaN,0.97,NaN,None,170,234,0,...,0.0,0.5,0.00,0.00,1.0,3.29,NaN,NaN,0.0,Manhattan
90186,2,2025-01-01 23:33:52,2025-01-01 23:48:39,NaN,6.00,NaN,None,219,86,0,...,0.0,0.5,0.00,0.00,1.0,-1.84,NaN,NaN,0.0,Queens
90187,2,2025-01-01 23:34:59,2025-01-01 23:51:28,NaN,3.59,NaN,None,215,258,0,...,0.0,0.5,0.00,0.00,1.0,3.06,NaN,NaN,0.0,Queens


## Sintaxis básica de `INNER JOIN`

```sql
SELECT
    tabla1.columna1,
    tabla2.columna2
FROM tabla1
INNER JOIN tabla2
    ON tabla1.clave = tabla2.clave

## Ejercicio 2

Generar una tabla agregando a la tabla de viajes la información del barrio de partida, solo si esa información está disponible.

In [12]:
trips_con_barrio = pd.read_sql_query("""
SELECT
   trips.*, 
   zones.*
FROM trips
INNER JOIN zones
    ON trips.PULocationID = zones.LocationID
""", con)
trips_con_barrio

# Se perdió alguna fila?

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,...,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,LocationID,Borough,Zone,service_zone
0,1,2025-01-01 00:18:38,2025-01-01 00:26:59,1.0,1.60,1.0,N,229,237,1,...,0.00,1.0,18.00,2.5,0.0,0.0,229,Manhattan,Sutton Place/Turtle Bay North,Yellow Zone
1,1,2025-01-01 00:32:40,2025-01-01 00:35:13,1.0,0.50,1.0,N,236,237,1,...,0.00,1.0,12.12,2.5,0.0,0.0,236,Manhattan,Upper East Side North,Yellow Zone
2,1,2025-01-01 00:44:04,2025-01-01 00:46:01,1.0,0.60,1.0,N,141,141,1,...,0.00,1.0,12.10,2.5,0.0,0.0,141,Manhattan,Lenox Hill West,Yellow Zone
3,2,2025-01-01 00:14:27,2025-01-01 00:20:01,3.0,0.52,1.0,N,244,244,2,...,0.00,1.0,9.70,0.0,0.0,0.0,244,Manhattan,Washington Heights South,Boro Zone
4,2,2025-01-01 00:21:34,2025-01-01 00:25:06,3.0,0.66,1.0,N,244,116,2,...,0.00,1.0,8.30,0.0,0.0,0.0,244,Manhattan,Washington Heights South,Boro Zone
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
90184,2,2025-01-01 23:41:33,2025-01-01 23:58:46,NaN,6.09,NaN,None,197,36,0,...,0.00,1.0,4.44,NaN,NaN,0.0,197,Queens,Richmond Hill,Boro Zone
90185,2,2025-01-01 23:02:52,2025-01-01 23:08:01,NaN,0.97,NaN,None,170,234,0,...,0.00,1.0,3.29,NaN,NaN,0.0,170,Manhattan,Murray Hill,Yellow Zone
90186,2,2025-01-01 23:33:52,2025-01-01 23:48:39,NaN,6.00,NaN,None,219,86,0,...,0.00,1.0,-1.84,NaN,NaN,0.0,219,Queens,Springfield Gardens South,Boro Zone
90187,2,2025-01-01 23:34:59,2025-01-01 23:51:28,NaN,3.59,NaN,None,215,258,0,...,0.00,1.0,3.06,NaN,NaN,0.0,215,Queens,South Jamaica,Boro Zone


¿Se perdió alguna fila?

### Ejercicio 3

Para cada zona, calcular la cantidad total de viajes que comienzan en esa zona, solo en el caso de que haya algún viaje.

In [15]:
# Primero obtenemos el listado sin calcular cantidades

# Este es un caso de join one-to-many, para cada fila de la tabla de la izquierda (zonas) tengo varias filas en la tabla de la derecha (viajes)
# Un JOIN puede aumentar la cantidad de filas.

listado_de_viajes_por_zona = pd.read_sql_query("""
SELECT
    z.*, t.trip_distance
FROM zones AS z
INNER JOIN trips AS t
    ON t.PULocationID = z.LocationID
""", con)

In [14]:
listado_de_viajes_por_zona

,LocationID,Borough,Zone,service_zone,trip_distance
0,1,EWR,Newark Airport,EWR,0.00
1,1,EWR,Newark Airport,EWR,0.00
2,1,EWR,Newark Airport,EWR,0.00
3,1,EWR,Newark Airport,EWR,0.00
4,1,EWR,Newark Airport,EWR,0.00
...,...,...,...,...,...
90184,265,None,Outside of NYC,None,29.15
90185,265,None,Outside of NYC,None,47.05
90186,265,None,Outside of NYC,None,61.10
90187,265,None,Outside of NYC,None,94.88


In [ ]:
# Ahora agregamos.
# Frecuentemente agrupamos por columnas de la tabla izquierda,
# porque suele representar la entidad principal del resultado
viajes_por_zona = pd.read_sql_query("""

""", con)

In [ ]:
viajes_por_zona

### Ejercicio 4

Y si quiero la información para todas las zonas, aunque haya 0 viajes?

In [ ]:
# Funciona esto??
viajes_por_zona = pd.read_sql_query("""
SELECT
    z.*,
    COUNT(*) AS n_trips
FROM zones AS z
LEFT JOIN trips AS t
    ON t.PULocationID = z.LocationID
GROUP BY z.LocationID
""", con)

Pregunta: ¿qué opinan del `z.*`? Algunas implementaciones de SQL no permite esto en tablas agrupadas.

In [ ]:
viajes_por_zona

Está bien la respuesta? Por qué pasa esto? Cómo la podemos arreglar?

**Sugerencia:** tener en cuenta que count() solo cuenta valores no nulos.

## Sintaxis general para múltiples JOIN

```sql
SELECT
    tabla1.*,
    tabla2.columna AS columna_2,
    tabla3.columna AS columna_3
FROM tabla1

LEFT JOIN tabla2
    ON tabla1.clave_1 = tabla2.clave

LEFT JOIN tabla3
    ON tabla1.clave_2 = tabla3.clave

### Ejercicio 5

Generar una tabla agregando a la tabla de viajes la información del barrio de partida y barrio de llegada (sin perder ninguna fila)

In [16]:
# Modificamos el ejercicio 1
trips_con_barrio = pd.read_sql_query("""
SELECT
    t.*, z.Borough as barrio_partida
FROM trips AS t
LEFT JOIN zones AS z
    ON t.PULocationID = z.LocationID
""", con)
trips_con_barrio

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,...,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,barrio_partida
0,1,2025-01-01 00:18:38,2025-01-01 00:26:59,1.0,1.60,1.0,N,229,237,1,...,3.5,0.5,3.00,0.00,1.0,18.00,2.5,0.0,0.0,Manhattan
1,1,2025-01-01 00:32:40,2025-01-01 00:35:13,1.0,0.50,1.0,N,236,237,1,...,3.5,0.5,2.02,0.00,1.0,12.12,2.5,0.0,0.0,Manhattan
2,1,2025-01-01 00:44:04,2025-01-01 00:46:01,1.0,0.60,1.0,N,141,141,1,...,3.5,0.5,2.00,0.00,1.0,12.10,2.5,0.0,0.0,Manhattan
3,2,2025-01-01 00:14:27,2025-01-01 00:20:01,3.0,0.52,1.0,N,244,244,2,...,1.0,0.5,0.00,0.00,1.0,9.70,0.0,0.0,0.0,Manhattan
4,2,2025-01-01 00:21:34,2025-01-01 00:25:06,3.0,0.66,1.0,N,244,116,2,...,1.0,0.5,0.00,0.00,1.0,8.30,0.0,0.0,0.0,Manhattan
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
90184,2,2025-01-01 23:41:33,2025-01-01 23:58:46,NaN,6.09,NaN,None,197,36,0,...,0.0,0.5,0.00,0.00,1.0,4.44,NaN,NaN,0.0,Queens
90185,2,2025-01-01 23:02:52,2025-01-01 23:08:01,NaN,0.97,NaN,None,170,234,0,...,0.0,0.5,0.00,0.00,1.0,3.29,NaN,NaN,0.0,Manhattan
90186,2,2025-01-01 23:33:52,2025-01-01 23:48:39,NaN,6.00,NaN,None,219,86,0,...,0.0,0.5,0.00,0.00,1.0,-1.84,NaN,NaN,0.0,Queens
90187,2,2025-01-01 23:34:59,2025-01-01 23:51:28,NaN,3.59,NaN,None,215,258,0,...,0.0,0.5,0.00,0.00,1.0,3.06,NaN,NaN,0.0,Queens


# Ejercicio 6
Para cada barrio (Borough), calcular la distancia promedio de los viajes que empiezan en en ese barrio, el precio promedio y la cantidad de viajes.

In [ ]:
estadisticos = pd.read_sql_query("""

""", con)

In [ ]:
estadisticos